# ice9 premium tier

The premium tier runs everything in the basic tier plus three cloud models:
- **gemini** (Google)
- **gpt_nano** (OpenAI)
- **haiku** (Anthropic)

More models means more votes in the noun consensus, which means higher confidence in what the image actually contains. The cloud models also tend to produce richer, more accurate descriptions than the local models alone.

Allow up to 3 minutes — cloud API calls add latency on top of the local VLMs.

**Run the first cell once.** The rest of the notebook explores that single result.

**Before you start:**
- Set your API key: `export ICE9_API_KEY=ice9_...` in your terminal before launching Jupyter, or set it in the cell below.
- Install dependencies: `pip install ice9 Pillow`

In [ ]:
from ice9 import Ice9
from ice9.exceptions import AnalysisTimeoutError, PartialResultError

# Set your image path here
IMAGE = "path/to/your/image.jpg"

# If you didn't set ICE9_API_KEY in your environment, you can set it here instead:
# import os
# os.environ["ICE9_API_KEY"] = "ice9_..."

client = Ice9(timeout=180.0)

print("Submitting image — this may take up to 3 minutes...")
try:
    result = client.analyze(IMAGE, tier="premium")
except AnalysisTimeoutError:
    print("Timed out. The server may be under load — try again.")
    raise
except PartialResultError as e:
    print(f"Warning: some services failed: {e.result.services_failed}")
    result = e.result

print(f"Done. Image ID: {result.image_id}")
print(f"Services: {result.services_submitted}")

## Summary caption

A single description synthesised from all model outputs — local and cloud.

In [ ]:
if result.caption:
    print(result.caption)
else:
    print("No caption yet.")

## Validated nouns

With more models voting, `vote_count` is higher here than in the basic tier. The ones marked confirmed were found by Florence-2 in the image.

In [ ]:
if result.nouns is not None:
    for noun in result.nouns.validated:
        confirmed = "✓ confirmed" if noun["grounding_validated"] else ""
        print(f"{noun['canonical']:20s}  votes={noun['vote_count']}  {confirmed}")
else:
    print("No validated nouns yet.")

In [ ]:
# All consensus nouns including ones below the grounding threshold
if result.nouns is not None:
    for noun in result.nouns.consensus:
        print(f"{noun['canonical']:20s}  votes={noun['vote_count']}  confidence={noun['confidence']:.0%}")

## Cloud model outputs

These are the three models that premium adds on top of basic.

In [ ]:
for service_name in ("gemini", "gpt_nano", "haiku"):
    service = getattr(result.services, service_name)
    if service is not None and service.text:
        print(f"{service_name}:")
        print(f"  {service.text}")
        print()

## Local model outputs

The same local VLMs that run in the basic tier.

In [ ]:
for service_name in ("blip", "florence2", "moondream", "ollama", "qwen"):
    service = getattr(result.services, service_name)
    if service is not None and service.text:
        print(f"{service_name}:")
        print(f"  {service.text}")
        print()

## Object detection — YOLO

YOLO detects and locates objects in the image. Each detection has a label, a confidence score, and a bounding box.

In [ ]:
if result.yolo_v8 is not None:
    if result.yolo_v8.predictions:
        for obj in result.yolo_v8.predictions:
            print(f"{obj['label']:25s}  confidence={obj['confidence']:.0%}  bbox={obj['bbox']}")
    else:
        print("No objects detected.")

## Florence-2 bounding boxes

Florence-2 found these regions corresponding to the validated nouns.

In [ ]:
if result.nouns is not None and result.nouns.regions:
    for region in result.nouns.regions:
        label = region.get("label") or region.get("text") or "unknown"
        bbox = region.get("bbox") or region.get("quad_box")
        print(f"{label:25s}  {bbox}")
else:
    print("No regions available.")

## Content moderation — nudenet

In [ ]:
from ice9 import CENSOR_LABELS

flagged = result.nsfw_detections(labels=CENSOR_LABELS)

if flagged:
    for detection in flagged:
        print(f"{detection['label']}  confidence={detection['confidence']:.0%}")
else:
    print("No flagged detections.")

## Censoring the image

In [ ]:
from IPython.display import display

censored = result.moderation.censor(IMAGE, method="pixelate")
display(censored)

## Colors

In [ ]:
if result.colors is not None:
    print(result.colors.dominant)

## Full result as JSON

In [ ]:
print(result.to_json(indent=2))